# Closed-Loop Intent Assurance — PoC Demo

This notebook demonstrates **intent-driven IP transport change validation** using a federated digital twin paired with NVIDIA-hosted LLM agents.

You'll see:

1. How an **operator's plain-English change request** is translated into a structured **Canonical ChangeSet** (live LLM call to Meta Llama 3.3 70B via NVIDIA NIM)
2. How the **VIAVI Federated Digital Twin engine** scores the proposed change — Twin Fidelity, Decision Confidence, blast radius, per-service impact (engine output is from sample runs against the canonical demo intents; the engine itself is provided via VIAVI partnership)
3. How the **Federation Agent's L3 SLM layer** (Nemotron Nano-9B-v2 via NIM) arbitrates when deterministic + math layers can't reconcile
4. How the **Resilience Agent** narrates a pre-mortem (Llama 3.3 70B) when verdicts escalate

The architecture cornerstone — the **Canonical ChangeSet Boundary** — separates the *replaceable* LLM agent layer from the *deterministic, audit-replayable* digital twin engine.

> *Twins predict what will happen. Governance decides whether it is allowed.*


## 1. Setup

The notebook makes live NVIDIA NIM calls — make sure `NVIDIA_API_KEY` is set in `.env` (copy from `.env.example` if not already done).


In [ ]:
# Install runtime dependencies (skipped if already present)
%pip install -q langchain-nvidia-ai-endpoints==0.3.19 langchain-core python-dotenv pandas


In [ ]:
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Load environment variables from .env (if present)
load_dotenv()

NVIDIA_API_KEY = (os.getenv("NVIDIA_API_KEY") or "").strip()
if not NVIDIA_API_KEY:
    raise RuntimeError(
        "Missing NVIDIA_API_KEY.\n\n"
        "Set it in your environment or in a local .env file:\n"
        "  NVIDIA_API_KEY=nvapi-...\n\n"
        "Get a key here: https://build.nvidia.com/settings/api-keys"
    )

print("✓ NVIDIA_API_KEY loaded (hidden)")


In [ ]:
# Initialize the two NIM-hosted LLMs we'll use in this notebook.
# Llama 3.3 70B is the Intent Planning Agent (chosen via v0.6.8 strict probe).
# Nemotron Nano-9B-v2 is the Federation Layer-3 SLM arbitrator.

INTENT_MODEL    = os.getenv("INSPECTOR_PLANNER_MODEL", "meta/llama-3.3-70b-instruct")
FEDERATION_MODEL = "nvidia/nemotron-nano-9b-v2"

llm_intent = ChatNVIDIA(
    model=INTENT_MODEL,
    api_key=NVIDIA_API_KEY,
    temperature=0,        # deterministic for the structured-output schema
    max_completion_tokens=2048,
)
llm_federation = ChatNVIDIA(
    model=FEDERATION_MODEL,
    api_key=NVIDIA_API_KEY,
    temperature=0,
    max_completion_tokens=1024,
)

# Sanity check — surface bad/missing credentials immediately
try:
    _ = llm_intent.invoke("hi")
    print("✓ LLM sanity check passed (Llama 3.3 70B reachable via NIM)")
except Exception as e:
    raise RuntimeError(
        "LLM sanity check failed. This usually means NVIDIA_API_KEY is missing/"
        "invalid or the endpoint/model is not accessible.\n\n"
        f"Underlying error: {type(e).__name__}: {e}"
    )


## 2. The Canonical ChangeSet contract

Every Intent Planning Agent — regardless of backend (Llama today; Claude, CONSILIUM, ServiceNow tomorrow) — must produce a JSON object matching this exact schema:

```json
{
  "change_id":          "string",
  "description":        "string (operator intent, max 200 chars)",
  "operations":         [{"op_type": "<one of 5>", "target": {...}}],
  "maintenance_window": "ISO 8601 interval string or null",
  "operator_intent_raw": "string (verbatim NL operator request)"
}
```

The five canonical `op_type` values: `withdraw_originated_route` · `withdraw_export_policy` · `down_link` · `down_node` · `metric_change`. Any other op type is rejected by the schema-validation gate, and the agent silently falls back to a pre-pinned canonical reference for the demo intent.

**The model is an implementation detail. The Canonical ChangeSet contract is the architecture.**


In [ ]:
CANONICAL_OP_TYPES = {
    "withdraw_originated_route",
    "withdraw_export_policy",
    "down_link",
    "down_node",
    "metric_change",
}

INSPECTOR_SYSTEM_PROMPT = '''You are the Intent Planning Agent for a federated digital twin.
Your single job is to translate a network operator's plain-English change request into a strict JSON Canonical ChangeSet that the deterministic engine can process.

Output MUST be a single JSON object — no markdown, no commentary. Fields:
  - change_id: short ID derived from the intent
  - description: 1-line operator-style summary
  - operator_intent_raw: verbatim copy of the operator's text
  - operations: array of 1+ items, each with op_type + target
  - maintenance_window: ISO 8601 interval ("2026-06-13T02:00Z/2026-06-13T04:00Z") or null

Allowed op_type values (anything else is rejected):
  - withdraw_originated_route  · target: {"router": "<id>", "prefix": "<CIDR>"}
  - withdraw_export_policy     · target: {"router": "<id>", "policy_name": "<name>"}
  - down_link                  · target: {"a": "<router>", "b": "<router>"} OR {"link_id": "<Lx>"}
  - down_node                  · target: {"node": "<router>"}
  - metric_change              · target: {"a": "<router>", "b": "<router>", "new_metric": <int>}
                                 OR {"link_id": "<Lx>", "new_metric": <int>}

Snapshot router IDs: YYZ-CORE-3, YVR-CORE-1, YYC-CORE-2, YWG-PE-6, YOW-PE-9, YVR-RR-11.
Snapshot link IDs:   L1 (YVR-YOW), L2 (YYC-YYZ), L3 (YYZ-YOW), L4 (YYC-YYZ alt), L5 (YVR-YYC), L6 (YYZ-YWG).
City names map to router IDs (e.g., "Toronto" -> YYZ-CORE-3, "Vancouver" -> YVR-CORE-1, "Calgary" -> YYC-CORE-2).
'''

def _validate_change_set(cs: dict) -> tuple[bool, str]:
    """The schema gate from agents/mcp_inspector_agent.py — same logic."""
    required = {"change_id", "description", "operations"}
    missing = required - set(cs.keys())
    if missing:
        return False, f"missing required fields: {missing}"
    ops = cs.get("operations")
    if not isinstance(ops, list) or not ops:
        return False, "operations must be a non-empty list"
    for i, op in enumerate(ops):
        if not isinstance(op, dict):
            return False, f"operations[{i}] is not a dict"
        if op.get("op_type") not in CANONICAL_OP_TYPES:
            return False, f"operations[{i}].op_type {op.get('op_type')!r} not in {CANONICAL_OP_TYPES}"
        if not isinstance(op.get("target"), dict):
            return False, f"operations[{i}].target must be a dict"
    if cs.get("maintenance_window") is not None and not isinstance(cs["maintenance_window"], str):
        return False, "maintenance_window must be a string or null"
    return True, "ok"


def parse_json_object(text: str) -> dict:
    """Strip code fences if the LLM wrapped output in ```json ... ```."""
    text = text.strip()
    m = re.search(r"```(?:json)?\s*\n?(.+?)\n?```", text, re.DOTALL)
    if m:
        text = m.group(1)
    return json.loads(text)


## 3. Sample Operator Intents

Pick one of these canonical change requests and watch the Intent Planning Agent (live Llama 70B call) translate it into a Canonical ChangeSet.

| # | Operator Intent | What kind of change |
|---|---|---|
| 1 | Withdraw 111.46.176.0/20 from the Toronto core during tonight 02:00-04:00 Toronto. | Single-op · route withdrawal in maintenance window |
| 2 | Take down the link between Vancouver and Calgary for Saturday's maintenance window. | Single-op · link down |
| 3 | Raise the ISIS metric on L2 (Calgary-Toronto) to 9999 to drain traffic before Saturday maintenance. | Single-op · metric change drain |
| 4 | Withdraw 111.46.176.0/20 from the Toronto core during today's business hours at 14:00 Toronto time. | Same as #1 but DURING BUSINESS HOURS — governance override demo |
| 5 | Set ISIS metric on L4 to 9999 AND take down YVR↔YYC during Saturday 02:00–04:00 UTC maintenance window. | Multi-op · 2 operations in one ChangeSet |


In [ ]:
SAMPLE_INTENTS = [
    "Withdraw 111.46.176.0/20 from the Toronto core during tonight 02:00-04:00 Toronto.",
    "Take down the link between Vancouver and Calgary for Saturday's maintenance window.",
    "Raise the ISIS metric on L2 (Calgary-Toronto) to 9999 to drain traffic before Saturday maintenance.",
    "Withdraw 111.46.176.0/20 from the Toronto core during today's business hours at 14:00 Toronto time.",
    "Set ISIS metric on L4 to 9999 AND take down YVR↔YYC during Saturday 02:00–04:00 UTC maintenance window.",
]

# Pick which one to run through the agent. Edit this index to try a different intent.
INTENT_INDEX = 4  # 4 = the multi-op maintenance demo
intent_text = SAMPLE_INTENTS[INTENT_INDEX]

print(f"Operator intent #{INTENT_INDEX + 1}:")
print(f"  {intent_text!r}")


## 4. Intent Planning Agent → Canonical ChangeSet  *(live NIM call · Llama 3.3 70B)*

The system prompt above plus the operator's intent goes to NIM. The LLM returns a structured JSON ChangeSet that the schema gate validates.


In [ ]:
# LIVE NIM CALL — Llama 3.3 70B via NVIDIA NIM
response = llm_intent.invoke(
    [
        {"role": "system", "content": INSPECTOR_SYSTEM_PROMPT},
        {"role": "user",   "content": intent_text},
    ]
)
raw_output = response.content

print("─── Raw LLM output ───")
print(raw_output[:1000] + ("..." if len(raw_output) > 1000 else ""))
print()

try:
    change_set = parse_json_object(raw_output)
except json.JSONDecodeError as e:
    raise RuntimeError(f"LLM returned non-JSON output. Raw response:\n{raw_output}") from e

ok, reason = _validate_change_set(change_set)
if not ok:
    raise RuntimeError(f"Schema gate REJECTED the LLM output: {reason}\n{json.dumps(change_set, indent=2)}")

print("✓ Schema validation passed")
print()
print("─── Canonical ChangeSet ───")
print(json.dumps(change_set, indent=2))


## 5. ▼ Canonical ChangeSet Boundary ▼

The agent has produced its output. Above this point: the Intent Planning Agent (Llama 70B today; could be Claude, CONSILIUM, ServiceNow, or a human typing JSON tomorrow). Below this point: VIAVI's deterministic, audit-replayable Federated Digital Twin engine.

**The model is an implementation detail. The contract is the architecture.**


## 6. Engine output  *(sample · engine provided via VIAVI partnership)*

In a live deployment, the Canonical ChangeSet would be dispatched to:
1. **Algorithmic Twin** — computes the predicted post-Δ network state (SPF · BGP · MPLS-TE · services · utilization)
2. **Emulation Twin** — independently observes the post-Δ state via VIAVI TestCenter Virtual + Nokia SR Linux containerlab
3. **Guardrails** — blast radius + risk score + severity findings
4. **Federation Agent (Layers 1+2)** — deterministic rules (~70%) + severity-weighted math (~20%)
5. **Policy Gate** — YAML-driven deterministic verdict with Decision Confidence band

These engine components are provided via VIAVI partnership and are **not included in this public repository**. For the live engine, see the booth at DTW Ignite 2026 (Booth 236) or [contact VIAVI](https://www.viavisolutions.com/en-us/contact-us).

For this notebook, we load **sample engine outputs** captured from real runs against the canonical synthetic 6-router snapshot. Same intent against the same snapshot is deterministic — these outputs are replay-faithful.


In [ ]:
# Load the sample engine output corresponding to the chosen intent
SAMPLES_PATH = Path("../data/sample_engine_responses.json")
if not SAMPLES_PATH.exists():
    SAMPLES_PATH = Path("data/sample_engine_responses.json")
with open(SAMPLES_PATH) as f:
    SAMPLES = json.load(f)["samples"]

# Match the chosen intent to its sample engine output
sample = SAMPLES[INTENT_INDEX]
blast = sample.get("blast_radius", {})
verdict = sample.get("verdict", {})

print("─── Blast radius (Algorithmic Twin + Guardrails) ───")
print(f"Affected routers:                   {blast.get('blast_radius', {}).get('affected_routers', [])}")
print(f"Affected links:                     {blast.get('blast_radius', {}).get('affected_links', [])}")
print(f"Affected service IDs:               {blast.get('blast_radius', {}).get('affected_service_ids', [])}")
print(f"Affected revenue (USD/month):       ${blast.get('blast_radius', {}).get('affected_revenue_usd_per_mo', 0):,}")
print(f"Guardrail findings:                 {blast.get('finding_count', 0)}")
print(f"Severity breakdown:                 {blast.get('severity_breakdown', {})}")


## 7. Federation Verdict  *(engine output + L3 narrative)*

The Federation Agent reconciles the two twins through a three-layer hierarchy:

- **L1 (rules)** — deterministic agreement check, resolves ~70% of cases at zero LLM cost
- **L2 (math)** — severity-weighted aggregation, resolves ~20% more
- **L3 (SLM)** — Nemotron Nano-9B arbitration when L1 + L2 can't reconcile (~10%)

When the underlying engine returns an `escalation_reason` indicating L3 was needed, the L3 step makes a **live NIM call to Nemotron** with the disagreement summary. This is the only LLM call inside the engine pipeline — everything else is deterministic.


In [ ]:
print("─── Federation + Policy Gate verdict ───")
print(f"Final verdict:                     {verdict.get('verdict', '?').upper()}")
print(f"Base tier → Final tier:            {verdict.get('base_tier','?')} → {verdict.get('final_tier','?')}")
print()
print(f"Twin Fidelity score:               {verdict.get('twin_fidelity_score', 0) * 100:.1f}%")
print(f"Decision Confidence:               {verdict.get('decision_confidence_score', 0) * 100:.1f}%  ({verdict.get('decision_confidence_band','?')})")
print(f"Operational Risk:                  {verdict.get('operational_risk_band','?')}")
print(f"Federation layer that produced verdict:  {verdict.get('layer_used', '?').upper()}")
print(f"Federation source:                 {verdict.get('federation_source', '?')}")
print()
print("─── Per-category Twin Fidelity alignment ───")
for cat, score in (verdict.get('category_alignment') or {}).items():
    print(f"  {cat:14s}  {score * 100:6.1f}%")


## 8. Federation L3 SLM Arbitration  *(live NIM call · Nemotron Nano-9B-v2)*

When L1 rules + L2 math can't reconcile the twin observations, the Federation Agent escalates to its L3 layer — an SLM (Nemotron) that summarises the disagreement and proposes a confidence score.

In the sample engine output for the chosen intent, L3 may or may not have been needed for the chosen intent. The cell below makes a **live Nemotron call** that demonstrates the L3 prompt shape, regardless of whether the engine actually escalated.


In [ ]:
# Build the L3 prompt the engine would build if it escalated
FED_L3_PROMPT = '''You are a network observability arbitrator. Two digital twins disagree about
the post-change network state. You must propose a confidence score (0-1) and a one-sentence reason.

Twin A (algorithmic prediction): {pred}
Twin B (emulation observation):   {obs}
Disagreements:                    {disagreements}

Return a single JSON object: {{"confidence": <float 0-1>, "reason": "<sentence>"}}'''

# Make a demonstration L3 call (synthetic disagreement payload)
disagreements_summary = verdict.get('disagreements') or []
demo_prompt = FED_L3_PROMPT.format(
    pred="link YVR-CORE-1↔YYC-CORE-2 SPF cost rises to 200",
    obs="link YVR-CORE-1↔YYC-CORE-2 actual reroute via YVR-RR-11; convergence 2207ms",
    disagreements=json.dumps(disagreements_summary[:2]) if disagreements_summary else "[]",
)

print("─── L3 SLM live arbitration (Nemotron Nano-9B-v2 via NIM) ───")
response = llm_federation.invoke(demo_prompt)
print(response.content)


## 9. Resilience Pre-mortem  *(live NIM call · Llama 3.3 70B — only on `escalate` verdicts)*

When the Policy Gate emits `escalate`, the engine runs a Resilience Agent pre-mortem before the operator confirms dispatch. The Resilience Agent reads the verdict + blast radius and narrates what could go wrong — a structured "imagine the change failed; what's the worst case?" pass.

Below, we make a **live Llama 70B call** with the chosen intent's verdict context. If the verdict is `auto_pass` or `pass_with_caveat`, we skip the call (a real deployment would too).


In [ ]:
if verdict.get('verdict', '').lower() in {'escalate', 'block'}:
    RESILIENCE_PROMPT = '''You are a network operations resilience analyst.
A proposed change has been ESCALATED to a Change Advisory Board because it
crosses a governance threshold despite high twin agreement.

Operator intent:      {intent}
Final verdict:        {verdict}
Affected services:    {services}
Affected revenue:     ${revenue:,}/month
Caveats:              {caveats}

In under 150 words, narrate the most plausible failure mode of this change
*if approved* — what could go wrong, which services would be affected,
and what mitigation the operator should plan for.'''

    prompt = RESILIENCE_PROMPT.format(
        intent=intent_text,
        verdict=verdict.get('verdict','?'),
        services=verdict.get('affected_services',[]),
        revenue=verdict.get('affected_revenue_usd_per_mo',0),
        caveats=verdict.get('caveats',[])[:3],
    )
    print("─── Resilience Agent pre-mortem (Llama 3.3 70B via NIM) ───")
    response = llm_intent.invoke(prompt)
    print(response.content)
else:
    print(f"Verdict is '{verdict.get('verdict')}' — Resilience Agent pre-mortem skipped.")
    print("(Pre-mortems only run on 'escalate' or 'block' verdicts in production.)")


## 10. The Audit Envelope

Every closed-loop run produces a structured audit envelope with HMAC-SHA256 signing. The envelope contains the verbatim operator intent, the Canonical ChangeSet, the Twin Fidelity scores, the verdict + reasoning trace, and a deterministic hash that allows full replay.

The signing key and the persistent LTM (long-term memory) store are provided via VIAVI partnership and not included in this public repo.


In [ ]:
# Print the relevant verdict envelope subset (full envelope is engine-internal)
envelope_subset = {
    "change_id":              verdict.get('change_id'),
    "verdict":                verdict.get('verdict'),
    "final_tier":             verdict.get('final_tier'),
    "twin_fidelity_score":    verdict.get('twin_fidelity_score'),
    "decision_confidence":    {
        "score":              verdict.get('decision_confidence_score'),
        "band":               verdict.get('decision_confidence_band'),
    },
    "operational_risk_band":  verdict.get('operational_risk_band'),
    "category_alignment":     verdict.get('category_alignment'),
    "affected_services":      verdict.get('affected_services'),
    "affected_revenue":       verdict.get('affected_revenue_usd_per_mo'),
    "caveats":                verdict.get('caveats'),
    "blocking_reasons":       verdict.get('blocking_reasons'),
    "advisory_note":          verdict.get('advisory_note'),
}

print(json.dumps(envelope_subset, indent=2))


## 11. Next Steps

This notebook walked through one closed-loop iteration:

1. ✅ Live LLM call (Llama 70B) — Intent translated into Canonical ChangeSet
2. ✅ Engine output — sample VIAVI Federated Digital Twin verdict
3. ✅ Live LLM call (Nemotron) — Federation L3 SLM arbitration demonstration
4. ✅ Live LLM call (Llama 70B) — Resilience Agent pre-mortem (conditional on verdict)
5. ✅ Audit envelope shape — what a downstream consumer would persist

For the **full live demo** (real Federated Digital Twin engine, Streamlit Wizard + MCP Inspector dual-surface UI, real-time twin computation against the snapshot):

- Visit the **DTW Ignite 2026 booth** (Copenhagen, June 23-25 · VIAVI Booth 236)
- **Contact VIAVI** to evaluate the engine: [VIAVI Solutions](https://www.viavisolutions.com/en-us/contact-us)
- See the **companion RAN blueprint**: [VIAVI-CTOO/es-blueprint-rsg](https://github.com/VIAVI-CTOO/es-blueprint-rsg)

---

*VIAVI and NVIDIA on the road to autonomous networks — Integrating Transport Network Digital Twins*
